# MOSAIC Source & Identity Tracking Detector
**Detection-only.** Read-only against source data. Every finding requires steward review before any action.

## What this detects

This detector validates the three foundational identity and provenance columns that every MOSAIC table must carry — `context_id`, `data_source_id`, and at least one source record key (`patient_id` / `member_id` / `record_id`). These are the columns that enable tenant isolation, source provenance tracing, EMPI matching, and audit lineage.

| Column | Purpose | Scope |
|---|---|---|
| `context_id` | Tenant isolation key — UC row-level security anchor | Required on all shared (multi-tenant) Silver/Gold tables |
| `data_source_id` | Feed provenance — registered identifier of the originating source | Required on every Bronze/Silver/Gold row from external feeds |
| `patient_id` / `member_id` / `record_id` | Source record keys — trace back to originating row | At least one required per row; preserved as-received |
| `enterprise_id` | Athena master key — preserve, never transform | As received from Athena; must NOT appear in source key columns |
| `person_id` | EMPI-assigned — downstream spine key | Must NOT appear in source key columns; assigned by EMPI only |

## Detection approach

This is a **structural detector** — it checks column presence, null completeness, composite key uniqueness, and conflation signals. Unlike content detectors (dates, emails, phones), it works at the **table level**: for each table it inventories which required columns are present, profiles their completeness, and evaluates conflation risks.

**AI use:** AI classifies whether a table is multi-tenant/multi-source (determining which requirements apply), detects conflation patterns (context_id used as data_source_id), and identifies alias columns needing renaming.

| Cell | What it does |
|---|---|
| 1-2 | Overview and Rule Reference |
| 3 | Config / Widgets |
| 4 | Constants: column taxonomy, alias maps, standardization rules |
| 5 | Discovery: inventory all required columns and aliases per table |
| 6 | Profiler: null rates, distinct counts, composite key uniqueness, conflation signals |
| 7 | AI Classifier: shared/multi-source classification, conflation detection |
| 8 | Rule Engine: evaluate all requirements per table |
| 9 | Write Results |
| 10 | Summary: blocking → by rule → by table → EMPI risk → work queue |

> **Hard constraint:** UPDATE, MERGE, DELETE, CTAS on source tables are never executed.


# MOSAIC Source & Identity Tracking Detector — Rule Reference

**Detection-only.** No source data is modified.

---

## §context_id — tenant isolation (§1)
| Rule tag | What it detects | Required action |
|---|---|---|
| `MISSING_CONTEXT_ID` | Shared (multi-tenant) table has no context_id or accepted alias | Add context_id column; certification blocker on shared Silver/Gold |
| `CONTEXT_ID_ALIAS_PRESENT` | partner_id or historical data_source_id used as tenant key | Rename to context_id in new models; document alias in data dictionary |
| `CONTEXT_ID_NULL_ROWS` | context_id has NULL values on a shared table | NULL context_id violates tenant isolation; route to quarantine |
| `MISSING_ROW_FILTER` | Shared table context_id present but no UC row filter evidence detected | Attach row-level access policy before certification; never rely on app-side filtering |

## §data_source_id — feed provenance (§2)
| Rule tag | What it detects | Required action |
|---|---|---|
| `MISSING_DATA_SOURCE_ID` | Table has no data_source_id column | Add data_source_id; required on all Bronze/Silver/Gold from external feeds |
| `DATA_SOURCE_ID_NULL_ROWS` | data_source_id has NULL values | Required on every row; route NULL rows to quarantine |
| `DATA_SOURCE_UNREGISTERED` | data_source_id value not in the registered allowlist | Fail or quarantine; unknown values block load per §5 |
| `DATA_SOURCE_ID_DROPPED` | Silver/Gold table missing data_source_id (dropped during conformance) | Restore; carry data_source_id through all layers per §2 |
| `DATA_SOURCE_ALIAS_PRESENT` | data_source column present (Tuva conformance name) without a data_source_id companion | Ensure Mosaic data_source_id is preserved alongside Tuva data_source mapping |

## §source record keys — provenance keys (§3)
| Rule tag | What it detects | Required action |
|---|---|---|
| `MISSING_SOURCE_KEY` | None of patient_id / member_id / record_id present in table | Add applicable source key column(s) per §3; at least one required |
| `SOURCE_KEY_ALL_NULL_ROWS` | Rows where all present source key columns are simultaneously NULL | Route to quarantine; never fabricate a source key value |
| `SOURCE_KEY_HIGH_NULL` | > threshold % of rows have all source keys NULL simultaneously | Escalate; systematic source key loss is a certification blocker |
| `MEMBER_PATIENT_CONFLATED` | Feed likely carries both member and patient concepts but only one key column present | Add missing key column; do not assume member_id equals patient_id |
| `PERSON_ID_IN_SOURCE_KEY` | person_id or enterprise_id detected in source key column values | Remove EMPI-assigned keys from source columns; keep source keys intact |
| `SOURCE_KEY_UNIQUENESS_VIOLATED` | Composite (data_source_id + context_id + source_keys) is not unique | Investigate duplicates; flag for dedupe or lineage investigation |

## §conflation checks (§5)
| Rule tag | What it detects | Required action |
|---|---|---|
| `CONTEXT_DATA_SOURCE_CONFLATION` | context_id used as data_source_id or vice versa (cardinality/format mismatch) | Separate columns; use context_id for tenant, data_source_id for feed |
| `ENTERPRISE_ID_IN_SOURCE_KEY` | Athena enterprise_id detected in patient_id / member_id / record_id | Remove; keep enterprise_id in its own column; source keys must be as-received |

## §documentation (§6)
| Rule tag | What it detects | Required action |
|---|---|---|
| `LAYER_VIOLATION` | Table in Silver/Gold layer missing a column required per §4 matrix | Add required column; certification blocker |
| `ALIAS_NOT_DOCUMENTED` | Accepted alias present (partner_id, data_source) but standard name absent | Document alias in data dictionary; plan migration to standard name |

---

## Enforcement
- Missing context_id on shared Silver/Gold tables and missing data_source_id on any layer are **certification blockers**.
- NULL context_id, NULL data_source_id, or all-NULL source keys route to quarantine — never fabricate values.
- Tenant isolation via UC row filters is mandatory; app-side filtering alone is insufficient.


In [0]:
import re, json, uuid
from datetime import datetime
from typing import Any, Dict, List, Tuple, Set, Optional
from collections import defaultdict
from pyspark.sql import functions as F, DataFrame

def _w(name: str, default) -> str:
    try:
        dbutils.widgets.text(name, str(default))
        return dbutils.widgets.get(name)
    except Exception:
        return str(default)

CFG: Dict[str, Any] = {
    "catalog":      _w("catalog",      "data_classification_source"),
    "schema":       _w("schema",       "identity_tracking_detector"),
    "table_filter": _w("table_filter", ""),
    "skip_views":   _w("skip_views",   "true").strip().lower() == "true",

    # Layer -- drives which requirements apply per §4 matrix
    "layer": _w("layer", "Unknown"),   # Bronze | Silver | Gold | Unknown

    "sample_size": int(_w("sample_size", 100_000)),
    "seed":        int(_w("seed",        42)),

    # Thresholds
    # null_threshold: % of rows with all source keys NULL before SOURCE_KEY_HIGH_NULL fires
    "null_threshold":       float(_w("null_threshold",       "1.0")),
    # dup_threshold: % of rows that are duplicates on composite key before flagging
    "dup_threshold":        float(_w("dup_threshold",        "0.0")),
    # conflation_ratio: if context_id distinct/data_source_id distinct > this ratio,
    # cardinality mismatch suggests conflation
    "conflation_ratio":     float(_w("conflation_ratio",     "10.0")),

    # Registered data_source_id allowlist (comma-separated).
    # If empty, DATA_SOURCE_UNREGISTERED check is skipped.
    # Example: "caremore_claims,mpg_labs,mpg_eligibility,athena_patient"
    "registered_sources":  _w("registered_sources", ""),

    # Whether to treat the scanned schema as multi-tenant by default when AI
    # cannot determine (conservative = assume shared/multi-tenant)
    "assume_multitenant":   _w("assume_multitenant", "true").strip().lower() == "true",

    # AI
    "ai_model":  _w("ai_model",  "databricks-meta-llama-3-3-70b-instruct"),
    "enable_ai": _w("enable_ai", "true").strip().lower() == "true",

    # Results
    "out_catalog": _w("out_catalog", "data_classification_target"),
    "out_schema":  _w("out_schema",  "data_classification_output"),
    "out_table":   _w("out_table",   "source_tracking_findings"),
}

RUN_ID = str(uuid.uuid4())
RUN_TS = datetime.utcnow().isoformat()

# Parse registered sources
REGISTERED_SOURCES: Set[str] = {
    s.strip().lower() for s in CFG["registered_sources"].split(",")
    if s.strip()
}

print(f"Run              : {RUN_ID}")
print(f"Scope            : {CFG['catalog']}.{CFG['schema']}")
print(f"Layer            : {CFG['layer']}")
print(f"Assume multi-tenant: {CFG['assume_multitenant']}")
print(f"Registered sources : {sorted(REGISTERED_SOURCES) or '(allowlist check disabled)'}")
print(f"AI               : {'on -> ' + CFG['ai_model'] if CFG['enable_ai'] else 'off'}")
print(f"Results          : {CFG['out_catalog']}.{CFG['out_schema']}.{CFG['out_table']}")


In [0]:
# ---------------------------------------------------------------------------
# CONSTANTS — column taxonomy, alias maps, pattern detection
# ---------------------------------------------------------------------------

TAGS = {
    # §context_id
    "MISSING_CONTEXT_ID":               "§context_id",
    "CONTEXT_ID_ALIAS_PRESENT":         "§context_id",
    "CONTEXT_ID_NULL_ROWS":             "§context_id",
    "MISSING_ROW_FILTER":               "§context_id",
    # §data_source_id
    "MISSING_DATA_SOURCE_ID":           "§data_source_id",
    "DATA_SOURCE_ID_NULL_ROWS":         "§data_source_id",
    "DATA_SOURCE_UNREGISTERED":         "§data_source_id",
    "DATA_SOURCE_ID_DROPPED":           "§data_source_id",
    "DATA_SOURCE_ALIAS_PRESENT":        "§data_source_id",
    # §source record keys
    "MISSING_SOURCE_KEY":               "§source-record-keys",
    "SOURCE_KEY_ALL_NULL_ROWS":         "§source-record-keys",
    "SOURCE_KEY_HIGH_NULL":             "§source-record-keys",
    "MEMBER_PATIENT_CONFLATED":         "§source-record-keys",
    "PERSON_ID_IN_SOURCE_KEY":          "§source-record-keys",
    "SOURCE_KEY_UNIQUENESS_VIOLATED":   "§source-record-keys",
    # §conflation
    "CONTEXT_DATA_SOURCE_CONFLATION":   "§conflation",
    "ENTERPRISE_ID_IN_SOURCE_KEY":      "§conflation",
    # §documentation
    "LAYER_VIOLATION":                  "§documentation",
    "ALIAS_NOT_DOCUMENTED":             "§documentation",
}

SEVERITY: Dict[str, int] = {
    "MISSING_CONTEXT_ID":               10,
    "CONTEXT_ID_NULL_ROWS":             10,
    "SOURCE_KEY_ALL_NULL_ROWS":         10,
    "SOURCE_KEY_HIGH_NULL":             10,
    "MISSING_DATA_SOURCE_ID":            9,
    "DATA_SOURCE_ID_NULL_ROWS":          9,
    "LAYER_VIOLATION":                   9,
    "DATA_SOURCE_UNREGISTERED":          9,
    "PERSON_ID_IN_SOURCE_KEY":           9,
    "ENTERPRISE_ID_IN_SOURCE_KEY":       9,
    "SOURCE_KEY_UNIQUENESS_VIOLATED":    8,
    "CONTEXT_DATA_SOURCE_CONFLATION":    8,
    "MEMBER_PATIENT_CONFLATED":          7,
    "MISSING_SOURCE_KEY":                7,
    "DATA_SOURCE_ID_DROPPED":            7,
    "MISSING_ROW_FILTER":                6,
    "CONTEXT_ID_ALIAS_PRESENT":          5,
    "DATA_SOURCE_ALIAS_PRESENT":         4,
    "ALIAS_NOT_DOCUMENTED":              4,
}

# ── Canonical column names ────────────────────────────────────────────────────
CONTEXT_ID_CANONICAL   = "context_id"
DATA_SOURCE_ID_CANONICAL = "data_source_id"
SOURCE_KEY_COLS        = {"patient_id", "member_id", "record_id"}
ENTERPRISE_ID_COL      = "enterprise_id"
PERSON_ID_COL          = "person_id"

# ── Accepted aliases per §1 and §6 ───────────────────────────────────────────
CONTEXT_ID_ALIASES = {
    "partner_id",          # historical alias per §1
    "tenant_id",           # common variant
    "client_id",           # common variant
    "lob_id",              # line-of-business variant
    "rbe_id",              # RBE variant mentioned in procedure
    "org_id",              # organizational variant
}
DATA_SOURCE_ALIASES = {
    "data_source",         # Tuva Input Layer conformance column name
    "source_system",       # common variant
    "source_id",           # common variant
    "feed_id",             # common variant
    "feed_name",           # common variant
}

# ── Column name regex patterns ────────────────────────────────────────────────
RE_CONTEXT_ID = re.compile(
    r"\b(context_id|partner_id|tenant_id|client_id|lob_id|rbe_id|org_id)\b",
    re.I
)
RE_DATA_SOURCE_ID = re.compile(
    r"\b(data_source_id|data_source|source_system|source_id|feed_id|feed_name)\b",
    re.I
)
RE_SOURCE_KEY = re.compile(
    r"\b(patient_id|member_id|record_id)\b",
    re.I
)
RE_ENTERPRISE_ID = re.compile(r"\benterprise_id\b", re.I)
RE_PERSON_ID     = re.compile(r"\bperson_id\b", re.I)

# ── EMPI key pattern (used to detect person_id/enterprise_id in source key values)
# These are typically UUID-like or prefixed with EMPI-specific patterns.
# We check for shared distinct value patterns between person_id and source keys.
RE_EMPI_PATTERN = re.compile(
    r"^(empi_|enterprise_|person_|[0-9a-f]{8}-[0-9a-f]{4}-)", re.I
)

STANDARDIZATION_RULES: Dict[str, list] = {

    "MISSING_CONTEXT_ID": [
        {"option_key": "add_context_id",
         "label":      "Add context_id column to shared table",
         "sql":        "-- ALTER TABLE ... ADD COLUMN context_id STRING NOT NULL",
         "notes":      "Required on all shared multi-tenant Silver/Gold tables (§1). "
                       "Enforce via Unity Catalog row-level access policy. "
                       "Never rely on application-side filtering alone (§1)."},
    ],

    "CONTEXT_ID_ALIAS_PRESENT": [
        {"option_key": "rename_to_context_id",
         "label":      "Rename alias to context_id in new models",
         "sql":        "-- SILVER: <alias_col> AS context_id",
         "notes":      "Standard column name is context_id (§1). "
                       "partner_id / historical data_source_id are accepted aliases -- "
                       "document in data dictionary and plan migration to context_id."},
    ],

    "CONTEXT_ID_NULL_ROWS": [
        {"option_key": "quarantine_null_context",
         "label":      "Route rows with NULL context_id to quarantine",
         "sql":        "-- Route rows WHERE context_id IS NULL to quarantine.",
         "notes":      "NULL context_id on a shared table violates tenant isolation (§1, §5). "
                       "Shared tables must reject NULL context_id."},
    ],

    "MISSING_ROW_FILTER": [
        {"option_key": "attach_row_filter",
         "label":      "Attach UC row-level access policy before certification",
         "sql":        "-- Apply Unity Catalog row filter on context_id column.",
         "notes":      "Tenant isolation must be enforced via Unity Catalog row filters / "
                       "row access policies (§1). Never rely on application-side filtering alone."},
    ],

    "MISSING_DATA_SOURCE_ID": [
        {"option_key": "add_data_source_id",
         "label":      "Add data_source_id column with registered value",
         "sql":        "-- ALTER TABLE ... ADD COLUMN data_source_id STRING NOT NULL",
         "notes":      "Required on every Bronze/Silver/Gold row from external feeds (§2). "
                       "Values must exist in the data-source registry. "
                       "Do not drop during conformance (§2)."},
    ],

    "DATA_SOURCE_ID_NULL_ROWS": [
        {"option_key": "quarantine_null_source",
         "label":      "Route rows with NULL data_source_id to quarantine",
         "sql":        "-- Route rows WHERE data_source_id IS NULL to quarantine.",
         "notes":      "data_source_id required on every row (§2). "
                       "NULL values indicate provenance loss."},
    ],

    "DATA_SOURCE_UNREGISTERED": [
        {"option_key": "quarantine_unregistered",
         "label":      "Fail or quarantine rows with unregistered data_source_id",
         "sql":        "-- Route rows WHERE data_source_id NOT IN (<registry>) to quarantine.",
         "notes":      "Unknown data_source_id values fail the load or route to quarantine (§5). "
                       "Adding a new value requires steward sign-off and registry entry (§2)."},
    ],

    "DATA_SOURCE_ID_DROPPED": [
        {"option_key": "restore_data_source_id",
         "label":      "Restore data_source_id column; carry through all layers",
         "sql":        "-- Re-add data_source_id from Bronze source in Silver/Gold transform.",
         "notes":      "Carry data_source_id through Silver and Gold (§2). "
                       "It is the join back to provenance and a required EMPI matching context. "
                       "Do not drop during conformance."},
    ],

    "DATA_SOURCE_ALIAS_PRESENT": [
        {"option_key": "ensure_mosaic_source_id_preserved",
         "label":      "Ensure Mosaic data_source_id is preserved alongside Tuva data_source",
         "sql":        "-- Keep data_source_id; map to data_source at Tuva conformance layer only.",
         "notes":      "data_source is the Tuva Input Layer column name (§2). "
                       "Mosaic data_source_id must be preserved; map to Tuva data_source at conformance."},
    ],

    "MISSING_SOURCE_KEY": [
        {"option_key": "add_source_key",
         "label":      "Add applicable source key column(s) per feed grain",
         "sql":        "-- Add patient_id STRING, member_id STRING, and/or record_id STRING "
                       "as applicable; at least one required per row (§3).",
         "notes":      "At least one of patient_id / member_id / record_id required (§3). "
                       "Populate all columns the feed provides. "
                       "Declare in data dictionary which columns apply."},
    ],

    "SOURCE_KEY_ALL_NULL_ROWS": [
        {"option_key": "quarantine_keyless_rows",
         "label":      "Route rows with all source keys NULL to quarantine",
         "sql":        "-- Route rows WHERE patient_id IS NULL AND member_id IS NULL "
                       "AND record_id IS NULL to quarantine.",
         "notes":      "Never fabricate a source key value (§3). "
                       "If source omitted all applicable keys, route to quarantine."},
    ],

    "SOURCE_KEY_HIGH_NULL": [
        {"option_key": "investigate_source_key_loss",
         "label":      "Investigate systematic source key loss",
         "sql":        "-- No transform. Investigate source feed and pipeline mapping.",
         "notes":      "High rate of all-NULL source keys indicates systematic provenance loss. "
                       "Certification blocker (§5). Steward must approve before Gold promotion."},
    ],

    "MEMBER_PATIENT_CONFLATED": [
        {"option_key": "add_missing_key_column",
         "label":      "Add missing member_id or patient_id column",
         "sql":        "-- ALTER TABLE ... ADD COLUMN member_id STRING -- or patient_id STRING",
         "notes":      "Member != patient (§3). When feed carries both, keep both columns. "
                       "Do not assume member_id equals patient_id."},
    ],

    "PERSON_ID_IN_SOURCE_KEY": [
        {"option_key": "remove_empi_key_from_source_col",
         "label":      "Remove EMPI-assigned key from source key column",
         "sql":        "-- Restore source key from Bronze; store person_id in its own column.",
         "notes":      "Do not put enterprise_id or person_id in source record key columns (§3). "
                       "person_id is assigned by EMPI downstream; source keys stay intact for audit."},
    ],

    "SOURCE_KEY_UNIQUENESS_VIOLATED": [
        {"option_key": "investigate_duplicates",
         "label":      "Investigate duplicate composite keys",
         "sql":        "-- SELECT data_source_id, context_id, patient_id, member_id, record_id, "
                       "COUNT(*) FROM tbl GROUP BY 1,2,3,4,5 HAVING COUNT(*) > 1",
         "notes":      "Source record uniqueness on composite key required (§5). "
                       "Duplicates flagged for dedupe or lineage investigation. "
                       "This composite is the EMPI matching input."},
    ],

    "CONTEXT_DATA_SOURCE_CONFLATION": [
        {"option_key": "separate_context_and_source",
         "label":      "Separate context_id (tenant) from data_source_id (feed)",
         "sql":        "-- Add context_id STRING for tenant; keep data_source_id for feed provenance.",
         "notes":      "Do not overload data_source_id with tenant meaning (§2). "
                       "Use context_id for tenant isolation, data_source_id for feed provenance."},
    ],

    "ENTERPRISE_ID_IN_SOURCE_KEY": [
        {"option_key": "separate_enterprise_id",
         "label":      "Move enterprise_id to its own column; restore source keys",
         "sql":        "-- ADD COLUMN enterprise_id STRING; restore patient_id/member_id from Bronze.",
         "notes":      "Athena enterprise_id must NOT be stored in source key columns (§3). "
                       "Preserve Athena enterprise_id in its own column when the feed provides it."},
    ],

    "LAYER_VIOLATION": [
        {"option_key": "add_required_column",
         "label":      "Add required column per §4 layer matrix",
         "sql":        "-- Add missing required column before promoting to this layer.",
         "notes":      "Layer violations block certification (§5). "
                       "See §4 required-column matrix for layer-specific requirements."},
    ],

    "ALIAS_NOT_DOCUMENTED": [
        {"option_key": "document_alias",
         "label":      "Document alias in data dictionary",
         "sql":        "-- No transform. Document accepted alias in data dictionary.",
         "notes":      "Accepted aliases must be declared in the data dictionary (§6). "
                       "Plan migration to standard column name in new models."},
    ],
}

print(f"Constants loaded: {len(TAGS)} tags | {len(STANDARDIZATION_RULES)} standardization entries")
print(f"Context ID aliases  : {sorted(CONTEXT_ID_ALIASES)}")
print(f"Data source aliases : {sorted(DATA_SOURCE_ALIASES)}")


In [0]:
# ---------------------------------------------------------------------------
# DISCOVERY:
# For each table, inventory which required tracking columns are present,
# which are absent, and which are present under alias names.
# Builds: table_inventory -- per table: which columns map to which roles.
# ---------------------------------------------------------------------------

cat, sch = CFG["catalog"], CFG["schema"]

def _esc(name: str) -> str:
    return name.replace("`", "``")

view_clause = "AND table_type IN ('MANAGED', 'EXTERNAL')" if CFG["skip_views"] else ""
tables: List[str] = [
    r.table_name for r in spark.sql(f"""
        SELECT table_name FROM `{_esc(cat)}`.information_schema.tables
        WHERE  table_schema = '{_esc(sch)}' {view_clause}
        ORDER BY table_name
    """).collect()
]

if CFG["table_filter"].strip():
    pat = re.compile(CFG["table_filter"], re.I)
    tables = [t for t in tables if pat.search(t)]

if not tables:
    raise ValueError(f"No tables found in {cat}.{sch}")

tbl_in = ", ".join(f"'{_esc(t)}'" for t in tables)
table_all_cols: Dict[str, List[Tuple[str, str]]] = defaultdict(list)
for r in spark.sql(f"""
    SELECT table_name, column_name, data_type
    FROM `{_esc(cat)}`.information_schema.columns
    WHERE table_schema = '{_esc(sch)}' AND table_name IN ({tbl_in})
    ORDER BY table_name, ordinal_position
""").collect():
    table_all_cols[r.table_name].append((r.column_name, r.data_type.upper()))


def _inventory_table(tbl: str, cols: List[Tuple[str, str]]) -> dict:
    col_lower = {c.lower(): c for c, _ in cols}

    def _find(canonical: str, aliases: Set[str]) -> Tuple[Optional[str], bool]:
        # Returns (actual_col_name, is_alias)
        if canonical in col_lower:
            return col_lower[canonical], False
        for alias in aliases:
            if alias in col_lower:
                return col_lower[alias], True
        return None, False

    ctx_col,  ctx_alias  = _find(CONTEXT_ID_CANONICAL,    CONTEXT_ID_ALIASES)
    dsrc_col, dsrc_alias = _find(DATA_SOURCE_ID_CANONICAL, DATA_SOURCE_ALIASES)

    # Source key columns: find all present
    src_key_present = {
        key: col_lower[key]
        for key in SOURCE_KEY_COLS
        if key in col_lower
    }

    # Tuva data_source conformance column (different from data_source_id)
    tuva_datasource = col_lower.get("data_source")

    # enterprise_id and person_id
    ent_col    = col_lower.get(ENTERPRISE_ID_COL)
    person_col = col_lower.get(PERSON_ID_COL)

    # Check for MEMBER_PATIENT_CONFLATED signal:
    # Both member_id and patient_id should be present when a feed carries both.
    # This heuristic fires when exactly ONE of the two is present -- AI refines this.
    has_member  = "member_id"  in col_lower
    has_patient = "patient_id" in col_lower
    possible_conflation = has_member != has_patient  # XOR: one present, other absent

    return {
        "all_cols":          [c for c, _ in cols],
        "col_dtypes":        {c: dt for c, dt in cols},
        # context_id
        "ctx_col":           ctx_col,
        "ctx_is_alias":      ctx_alias,
        # data_source_id
        "dsrc_col":          dsrc_col,
        "dsrc_is_alias":     dsrc_alias,
        "has_tuva_datasource": tuva_datasource is not None,
        # source keys
        "src_key_present":   src_key_present,   # {role: actual_col_name}
        # special columns
        "ent_col":           ent_col,
        "person_col":        person_col,
        # signals
        "possible_key_conflation": possible_conflation,
        # to be filled by profiler and AI
        "is_multitenant":    CFG["assume_multitenant"],  # AI may override
        "is_multisource":    True,   # conservative default; AI may override
        "ai_confidence":     "low",
        "total":             0,
        "ctx_null":          0,
        "dsrc_null":         0,
        "dsrc_distinct_vals":{},
        "src_key_all_null":  0,
        "ctx_distinct":      0,
        "dsrc_distinct":     0,
        "composite_dup":     0,
        "conflation_signal": False,
        "person_in_src_key": False,
    }


table_inventory: Dict[str, dict] = {}
for tbl in tables:
    cols = table_all_cols.get(tbl, [])
    if not cols:
        continue
    inv = _inventory_table(tbl, cols)
    table_inventory[tbl] = inv

print(f"Scope   : {cat}.{sch}")
print(f"Tables  : {len(tables)}")
print(f"Layer   : {CFG['layer']}")
print()
for tbl, inv in table_inventory.items():
    ctx  = inv['ctx_col'] or "(absent)"
    dsrc = inv['dsrc_col'] or "(absent)"
    keys = list(inv['src_key_present'].values()) or ["(absent)"]
    alias_flags = []
    if inv['ctx_is_alias']:  alias_flags.append(f"ctx=alias:{inv['ctx_col']}")
    if inv['dsrc_is_alias']: alias_flags.append(f"dsrc=alias:{inv['dsrc_col']}")
    print(f"  {tbl}")
    print(f"    context_id    : {ctx}  data_source_id: {dsrc}  src_keys: {keys}")
    if alias_flags: print(f"    aliases       : {alias_flags}")


In [0]:
# ---------------------------------------------------------------------------
# PROFILER -- one SQL per table for ALL required tracking column stats:
# - null counts for context_id, data_source_id, source keys
# - distinct counts for context_id and data_source_id (for conflation check)
# - distinct values of data_source_id (for registry check)
# - count of rows where ALL source keys are simultaneously NULL
# - count of approximate composite key duplicates
# - person_id value overlap with source key columns (conflation signal)
# ---------------------------------------------------------------------------

def _esc(name: str) -> str:
    return name.replace("`", "``")

def get_sample(tbl: str) -> DataFrame:
    fq = f"`{_esc(cat)}`.`{_esc(sch)}`.`{_esc(tbl)}`"
    n  = CFG["sample_size"]
    try:
        return spark.sql(f"SELECT * FROM {fq} TABLESAMPLE ({n} ROWS)")
    except Exception:
        total = spark.sql(f"SELECT COUNT(*) AS n FROM {fq}").collect()[0]["n"]
        return (
            spark.table(fq)
            .sample(withReplacement=False, fraction=min(1.0, n / max(1, total)), seed=CFG["seed"])
            .limit(n)
        )


def profile_table(tbl: str, inv: dict) -> None:
    fq    = f"`{_esc(cat)}`.`{_esc(sch)}`.`{_esc(tbl)}`"
    exprs = ["COUNT(*) AS __total__"]

    # context_id null and distinct
    if inv["ctx_col"]:
        cs = f"`{_esc(inv['ctx_col'])}`"
        exprs += [
            f"COUNT(*) - COUNT({cs}) AS __ctx_null__",
            f"APPROX_COUNT_DISTINCT({cs}) AS __ctx_distinct__",
        ]

    # data_source_id null and distinct
    if inv["dsrc_col"]:
        cs = f"`{_esc(inv['dsrc_col'])}`"
        exprs += [
            f"COUNT(*) - COUNT({cs}) AS __dsrc_null__",
            f"APPROX_COUNT_DISTINCT({cs}) AS __dsrc_distinct__",
        ]

    # all source keys simultaneously NULL
    if inv["src_key_present"]:
        null_conditions = " AND ".join(
            f"`{_esc(col)}` IS NULL"
            for col in inv["src_key_present"].values()
        )
        exprs.append(
            f"COUNT(CASE WHEN {null_conditions} THEN 1 END) AS __src_all_null__"
        )

    # composite key duplicate count (data_source_id + context_id + source keys)
    # Only when all three pillars present
    composite_parts = []
    if inv["dsrc_col"]:  composite_parts.append(f"`{_esc(inv['dsrc_col'])}`")
    if inv["ctx_col"]:   composite_parts.append(f"`{_esc(inv['ctx_col'])}`")
    for col in inv["src_key_present"].values():
        composite_parts.append(f"`{_esc(col)}`")

    if len(composite_parts) >= 2:
        group_key = ", ".join(composite_parts)
        # Count rows that belong to groups with count > 1 (approximate via subquery)
        # Use a simple approach: total - count_distinct_composite
        exprs.append(
            f"COUNT(*) - APPROX_COUNT_DISTINCT(CONCAT_WS('|', {group_key})) AS __composite_dup__"
        )

    try:
        row   = spark.sql(f"SELECT {', '.join(exprs)} FROM {fq}").collect()[0].asDict()
        total = row.get("__total__", 0) or 0
        inv.update({
            "total":          total,
            "ctx_null":       int(row.get("__ctx_null__",      0) or 0),
            "ctx_distinct":   int(row.get("__ctx_distinct__",  0) or 0),
            "dsrc_null":      int(row.get("__dsrc_null__",     0) or 0),
            "dsrc_distinct":  int(row.get("__dsrc_distinct__", 0) or 0),
            "src_key_all_null": int(row.get("__src_all_null__", 0) or 0),
            "composite_dup":  max(0, int(row.get("__composite_dup__", 0) or 0)),
        })
    except Exception as e:
        print(f"  [WARN] Profile failed for {tbl}: {e}")

    # Collect distinct data_source_id values for registry check
    if inv["dsrc_col"]:
        cs = f"`{_esc(inv['dsrc_col'])}`"
        try:
            rows = spark.sql(f"""
                SELECT LOWER(TRIM({cs})) AS val, COUNT(*) AS cnt
                FROM `{_esc(cat)}`.`{_esc(sch)}`.`{_esc(tbl)}`
                WHERE {cs} IS NOT NULL
                GROUP BY 1 ORDER BY cnt DESC LIMIT 100
            """).collect()
            inv["dsrc_distinct_vals"] = {r["val"]: int(r["cnt"]) for r in rows}
        except Exception:
            inv["dsrc_distinct_vals"] = {}

    # Conflation signal: cardinality mismatch between context_id and data_source_id
    # If context_id has very few distinct values (tenants) and data_source_id has many,
    # or vice versa, they may be correctly separated. But if they have similar cardinality
    # in a range that fits tenant counts (2-20), suspect conflation.
    ctx_d  = inv.get("ctx_distinct",  0)
    dsrc_d = inv.get("dsrc_distinct", 0)
    if ctx_d > 0 and dsrc_d > 0:
        ratio = max(ctx_d, dsrc_d) / max(1, min(ctx_d, dsrc_d))
        # Conflation risk when they have very similar cardinality (ratio close to 1)
        # AND both are in the low-cardinality range typical of tenant counts (2-50)
        if ratio <= CFG["conflation_ratio"] and ctx_d <= 50 and dsrc_d <= 50:
            inv["conflation_signal"] = True

    # Check whether person_id values appear in source key columns
    # (signal that EMPI key was stored in source key column)
    if inv["person_col"] and inv["src_key_present"]:
        try:
            # Sample 100 person_id values and check overlap with source keys
            person_vals = {
                r["v"] for r in spark.sql(f"""
                    SELECT DISTINCT `{_esc(inv['person_col'])}` AS v
                    FROM `{_esc(cat)}`.`{_esc(sch)}`.`{_esc(tbl)}`
                    WHERE `{_esc(inv['person_col'])}` IS NOT NULL LIMIT 100
                """).collect()
            }
            for key_col in inv["src_key_present"].values():
                src_key_vals = {
                    r["v"] for r in spark.sql(f"""
                        SELECT DISTINCT `{_esc(key_col)}` AS v
                        FROM `{_esc(cat)}`.`{_esc(sch)}`.`{_esc(tbl)}`
                        WHERE `{_esc(key_col)}` IS NOT NULL LIMIT 100
                    """).collect()
                }
                overlap = person_vals & src_key_vals
                if len(overlap) > 0:
                    inv["person_in_src_key"] = True
                    break
        except Exception:
            pass


# Run profiler
table_samples: Dict[str, DataFrame] = {}
for tbl, inv in table_inventory.items():
    sample_df = get_sample(tbl)
    table_samples[tbl] = sample_df
    profile_table(tbl, inv)

print("Profiler done.")
for tbl, inv in table_inventory.items():
    total = inv.get("total", 0)
    print(f"  {tbl}: {total:,} rows | "
          f"ctx_null={inv.get('ctx_null',0)} dsrc_null={inv.get('dsrc_null',0)} "
          f"src_all_null={inv.get('src_key_all_null',0)} "
          f"composite_dup={inv.get('composite_dup',0)} "
          f"conflation={inv.get('conflation_signal',False)}")


In [0]:
# ---------------------------------------------------------------------------
# AI CLASSIFIER -- classifies each table as multi-tenant / multi-source,
# detects conflation, and refines member/patient key presence assessment.
# ---------------------------------------------------------------------------

BATCH_SIZE = 10  # tables per call (table context is richer than column context)

def _ai_query(prompt: str) -> str:
    safe = prompt.replace("'", "''")
    raw  = spark.sql(
        f"SELECT ai_query('{CFG['ai_model']}', '{safe}', failOnError => false) AS r"
    ).collect()[0]["r"]
    if hasattr(raw, "errorStatus") and raw.errorStatus:
        raise ValueError(raw.errorStatus)
    return raw.result if hasattr(raw, "result") else str(raw)

def _chunked(items: list, size: int = BATCH_SIZE):
    for i in range(0, len(items), size):
        yield items[i:i + size]


def classify_tables(tbls: List[str]) -> None:
    if not CFG["enable_ai"] or not tbls:
        return

    for chunk in _chunked(tbls):
        payload = json.dumps([{
            "table":             tbl,
            "columns":           table_inventory[tbl]["all_cols"][:40],
            "has_context_id":    table_inventory[tbl]["ctx_col"] is not None,
            "ctx_col_name":      table_inventory[tbl]["ctx_col"],
            "ctx_is_alias":      table_inventory[tbl]["ctx_is_alias"],
            "has_data_source_id":table_inventory[tbl]["dsrc_col"] is not None,
            "dsrc_col_name":     table_inventory[tbl]["dsrc_col"],
            "dsrc_distinct":     table_inventory[tbl].get("dsrc_distinct", 0),
            "dsrc_sample_values":list(table_inventory[tbl].get("dsrc_distinct_vals", {}).keys())[:10],
            "ctx_distinct":      table_inventory[tbl].get("ctx_distinct", 0),
            "source_keys_present":list(table_inventory[tbl]["src_key_present"].keys()),
            "has_member_id":     "member_id"  in table_inventory[tbl]["src_key_present"],
            "has_patient_id":    "patient_id" in table_inventory[tbl]["src_key_present"],
            "conflation_signal": table_inventory[tbl].get("conflation_signal", False),
            "total_rows":        table_inventory[tbl].get("total", 0),
        } for tbl in chunk if tbl in table_inventory], default=str)

        prompt = (
            "Healthcare data governance expert. Reply ONLY with a JSON array -- no prose, no markdown. "
            "For each table determine: "
            "(1) is_multitenant: true if this table holds data from more than one tenant/partner/RBE "
            "or if context_id / a tenant key is clearly present. "
            "False only if this is clearly a single-tenant internal table. "
            "(2) is_multisource: true if this table blends data from multiple source systems/feeds, "
            "or if data_source_id has > 1 distinct value. "
            "(3) context_data_source_conflation: true if context_id and data_source_id appear to "
            "store the same kind of value (tenant IDs used as source IDs or vice versa). "
            "Signals: similar cardinality (both 2-20 distinct), similar value patterns. "
            "(4) member_patient_feed: 'member_only', 'patient_only', 'both', or 'neither'. "
            "Use column names and table name to infer. "
            "(5) confidence: high/medium/low. "
            f"Tables: {payload}. "
            'Return: [{"table":"<t>","is_multitenant":true|false,"is_multisource":true|false,'
            '"context_data_source_conflation":false,"member_patient_feed":"<m>",'
            '"confidence":"high|medium|low"}]'
        )
        try:
            for item in json.loads(_ai_query(prompt)):
                tbl = item.get("table", "")
                if tbl not in table_inventory:
                    continue
                table_inventory[tbl].update({
                    "is_multitenant":             item.get("is_multitenant",   CFG["assume_multitenant"]),
                    "is_multisource":             item.get("is_multisource",   True),
                    "conflation_confirmed":        item.get("context_data_source_conflation", False),
                    "member_patient_feed":         item.get("member_patient_feed", "neither"),
                    "ai_confidence":               item.get("confidence", "low"),
                })
        except Exception as e:
            print(f"  [WARN] AI classification failed for chunk: {e}")
            for tbl in chunk:
                if tbl in table_inventory:
                    table_inventory[tbl].update({
                        "is_multitenant":    CFG["assume_multitenant"],
                        "is_multisource":    True,
                        "conflation_confirmed": False,
                        "member_patient_feed": "neither",
                        "ai_confidence":     "low",
                        "ai_error":          str(e),
                    })


classify_tables(list(table_inventory.keys()))

print("AI classification done.")
for tbl, inv in table_inventory.items():
    print(f"  {tbl}: multitenant={inv.get('is_multitenant','?')} "
          f"multisource={inv.get('is_multisource','?')} "
          f"feed={inv.get('member_patient_feed','?')} "
          f"conflation={inv.get('conflation_confirmed','?')} "
          f"conf={inv.get('ai_confidence','?')}")


In [0]:
# ---------------------------------------------------------------------------
# RULE ENGINE -- one _check_table() per table.
# All stats from pre-computed table_inventory dict.
# ---------------------------------------------------------------------------

def _esc(name: str) -> str:
    return name.replace("`", "``")


def _finding(tbl, col, tag, count, total, samples, action,
             hint=None, confidence="high", std_options=None,
             auto_decided_action=None) -> dict:
    pct     = round(count / total * 100, 4) if total else 0.0
    options = std_options if std_options is not None else STANDARDIZATION_RULES.get(tag, [])
    decided = auto_decided_action if auto_decided_action is not None else None
    return {
        "run_id":                   RUN_ID,
        "run_ts":                   RUN_TS,
        "catalog":                  cat,
        "schema":                   sch,
        "table_name":               tbl,
        "column_name":              col,
        "layer":                    CFG["layer"],
        "rule_ref":                 TAGS.get(tag, "§?"),
        "classification":           tag,
        "affected_count":           int(count),
        "affected_pct":             float(pct),
        "total_rows":               int(total),
        "sample_values":            json.dumps(samples, default=str),
        "recommended_action":       action,
        "dictionary_strategy_hint": hint,
        "standardization_rule":     json.dumps(options, ensure_ascii=False),
        "confidence":               confidence,
        "needs_steward_review":     decided is None,
        "decided_action":           decided,
        "decided_by":               None,
    }


def _check_table(tbl: str, inv: dict) -> list:
    total         = inv.get("total", 0)
    is_mt         = inv.get("is_multitenant",  CFG["assume_multitenant"])
    is_ms         = inv.get("is_multisource",  True)
    layer         = CFG["layer"]
    ctx_col       = inv.get("ctx_col")
    dsrc_col      = inv.get("dsrc_col")
    src_keys      = inv.get("src_key_present", {})
    confidence    = inv.get("ai_confidence", "low")
    findings      = []

    # ── §context_id checks ────────────────────────────────────────────────────

    # MISSING_CONTEXT_ID: shared table without context_id
    if is_mt and not ctx_col and layer in ("Silver", "Gold", "Unknown"):
        findings.append(_finding(tbl, "context_id", "MISSING_CONTEXT_ID",
            total, total, [],
            f"Table is multi-tenant but has no context_id column (or accepted alias). "
            "Required on all shared Silver/Gold tables for UC row-level isolation (§1). "
            "Certification blocker.",
            confidence="high" if confidence == "high" else "medium",
        ))

    # CONTEXT_ID_ALIAS_PRESENT: using partner_id or historical alias
    if ctx_col and inv.get("ctx_is_alias"):
        findings.append(_finding(tbl, ctx_col, "CONTEXT_ID_ALIAS_PRESENT",
            0, total, [],
            f"Column '{ctx_col}' is used as tenant context key but is not the standard name. "
            "Standard name is context_id (§1). "
            "Document alias in data dictionary; use context_id in new models.",
            confidence="high",
            auto_decided_action="document_alias",
        ))

    # CONTEXT_ID_NULL_ROWS: NULLs on shared table
    if ctx_col and is_mt and inv.get("ctx_null", 0) > 0:
        findings.append(_finding(tbl, ctx_col, "CONTEXT_ID_NULL_ROWS",
            inv["ctx_null"], total, [],
            f"{inv['ctx_null']:,} row(s) have NULL context_id on a shared (multi-tenant) table. "
            "NULL context_id violates tenant isolation (§1, §5). "
            "Route to quarantine; shared tables must reject NULL context_id.",
            confidence="high",
        ))

    # LAYER_VIOLATION: Silver/Gold shared table without context_id
    if is_mt and not ctx_col and layer in ("Gold",):
        findings.append(_finding(tbl, "context_id", "LAYER_VIOLATION",
            total, total, [],
            f"Gold layer shared table is missing required context_id column (§4 matrix). "
            "Certification blocker.",
            confidence="high",
        ))

    # ── §data_source_id checks ─────────────────────────────────────────────────

    # MISSING_DATA_SOURCE_ID: any table from external feed
    if is_ms and not dsrc_col:
        tag = "DATA_SOURCE_ID_DROPPED" if layer in ("Silver", "Gold") else "MISSING_DATA_SOURCE_ID"
        findings.append(_finding(tbl, "data_source_id", tag,
            total, total, [],
            f"Table appears to be multi-source but has no data_source_id column. "
            + ("data_source_id may have been dropped during conformance (§2). " if tag == "DATA_SOURCE_ID_DROPPED" else "")
            + "Required on every Bronze/Silver/Gold row from external feeds (§2). "
            "Carry through all layers; it is the join back to provenance and EMPI matching context.",
            confidence="high" if confidence == "high" else "medium",
        ))

    # DATA_SOURCE_ID_NULL_ROWS
    if dsrc_col and inv.get("dsrc_null", 0) > 0:
        findings.append(_finding(tbl, dsrc_col, "DATA_SOURCE_ID_NULL_ROWS",
            inv["dsrc_null"], total, [],
            f"{inv['dsrc_null']:,} row(s) have NULL data_source_id. "
            "Required on every row (§2); NULL indicates provenance loss. "
            "Route to quarantine.",
            confidence="high",
        ))

    # DATA_SOURCE_ALIAS_PRESENT: Tuva data_source column without data_source_id
    if inv.get("has_tuva_datasource") and not dsrc_col:
        findings.append(_finding(tbl, "data_source", "DATA_SOURCE_ALIAS_PRESENT",
            0, total, [],
            "Table has Tuva data_source column but no Mosaic data_source_id column. "
            "Mosaic data_source_id must be preserved; map to Tuva data_source at conformance only (§2). "
            "Ensure data_source_id is not dropped before the Tuva mapping step.",
            confidence="medium",
        ))

    # DATA_SOURCE_UNREGISTERED: values not in registry
    if dsrc_col and REGISTERED_SOURCES and inv.get("dsrc_distinct_vals"):
        unregistered = [
            v for v in inv["dsrc_distinct_vals"]
            if v.strip() and v.lower() not in REGISTERED_SOURCES
        ]
        if unregistered:
            findings.append(_finding(tbl, dsrc_col, "DATA_SOURCE_UNREGISTERED",
                sum(inv["dsrc_distinct_vals"].get(v, 0) for v in unregistered),
                total, unregistered[:10],
                f"{len(unregistered)} data_source_id value(s) not in the registered allowlist: "
                f"{unregistered[:5]}. "
                "Unknown values fail the load or route to quarantine (§5). "
                "Adding a new value requires steward sign-off and registry entry (§2).",
                confidence="high",
            ))

    # ── §source record key checks ──────────────────────────────────────────────

    # MISSING_SOURCE_KEY: no source key columns at all
    if not src_keys:
        findings.append(_finding(tbl, "patient_id/member_id/record_id", "MISSING_SOURCE_KEY",
            total, total, [],
            "Table has no source record key columns (patient_id, member_id, record_id). "
            "At least one is required per row (§3). "
            "Add applicable key column(s); declare in data dictionary which columns apply.",
            confidence="high",
        ))

    # SOURCE_KEY_ALL_NULL_ROWS: rows where all present source keys are simultaneously NULL
    all_null_count = inv.get("src_key_all_null", 0)
    if src_keys and all_null_count > 0:
        null_pct = all_null_count / total * 100 if total else 0
        findings.append(_finding(tbl, "/".join(src_keys.values()), "SOURCE_KEY_ALL_NULL_ROWS",
            all_null_count, total, [],
            f"{all_null_count:,} row(s) ({null_pct:.1f}%) have NULL in all source key columns "
            f"({list(src_keys.values())}). "
            "Route to quarantine; never fabricate a source key value (§3).",
            confidence="high",
        ))
        # SOURCE_KEY_HIGH_NULL: rate threshold
        if null_pct > CFG["null_threshold"]:
            findings.append(_finding(tbl, "/".join(src_keys.values()), "SOURCE_KEY_HIGH_NULL",
                all_null_count, total, [],
                f"{null_pct:.1f}% of rows have all source keys NULL. "
                "Systematic source key loss is a certification blocker (§5). "
                "Investigate source feed and pipeline mapping.",
                confidence="high",
                auto_decided_action="investigate_source_key_loss",
            ))

    # MEMBER_PATIENT_CONFLATED: AI says feed has both but only one key column present
    feed_type = inv.get("member_patient_feed", "neither")
    if feed_type == "both":
        has_m = "member_id"  in src_keys
        has_p = "patient_id" in src_keys
        if not (has_m and has_p):
            missing = "member_id" if not has_m else "patient_id"
            findings.append(_finding(tbl, missing, "MEMBER_PATIENT_CONFLATED",
                0, total, [],
                f"Feed appears to carry both member and patient concepts but '{missing}' is absent. "
                "Member != patient (§3). Keep both columns when feed carries both. "
                "Do not assume member_id equals patient_id.",
                confidence=confidence,
            ))

    # PERSON_ID_IN_SOURCE_KEY: EMPI key detected in source key column values
    if inv.get("person_in_src_key"):
        findings.append(_finding(tbl, "/".join(src_keys.values()), "PERSON_ID_IN_SOURCE_KEY",
            0, total, [],
            "person_id values detected overlapping with source key column values. "
            "Do not put person_id or enterprise_id in source record key columns (§3). "
            "person_id is assigned by EMPI downstream; source keys must stay as-received for audit.",
            confidence="medium",
        ))

    # SOURCE_KEY_UNIQUENESS_VIOLATED: composite key has duplicates
    dup_count = inv.get("composite_dup", 0)
    if dup_count > 0 and CFG["dup_threshold"] == 0.0:
        findings.append(_finding(tbl, "composite_key", "SOURCE_KEY_UNIQUENESS_VIOLATED",
            dup_count, total, [],
            f"Approximately {dup_count:,} row(s) are duplicated on the composite key "
            f"(data_source_id + context_id + source record keys). "
            "This composite is the EMPI matching input and dedupe/lineage key (§3, §5). "
            "Investigate duplicates.",
            confidence="medium",
        ))

    # ── §conflation checks ─────────────────────────────────────────────────────

    # CONTEXT_DATA_SOURCE_CONFLATION
    if inv.get("conflation_confirmed") or (
        inv.get("conflation_signal") and not inv.get("conflation_confirmed") == False
    ):
        findings.append(_finding(tbl, f"{ctx_col or 'context_id'}/{dsrc_col or 'data_source_id'}",
            "CONTEXT_DATA_SOURCE_CONFLATION",
            0, total, [],
            "context_id and data_source_id appear to have similar cardinality and may store "
            "overlapping concepts (tenant IDs used as source IDs or vice versa). "
            "Do not overload data_source_id with tenant meaning (§2). "
            "Use context_id for tenant isolation, data_source_id for feed provenance.",
            confidence="medium",
        ))

    return findings


# ── Main loop ─────────────────────────────────────────────────────────────────
all_findings: List[dict] = []

for tbl, inv in table_inventory.items():
    findings  = _check_table(tbl, inv)
    all_findings.extend(findings)
    print(f"  ok {tbl}: {len(findings)} finding(s)")

print(f"\nRule engine done. {len(all_findings)} total finding(s).")


In [0]:
from pyspark.sql.types import (StructType, StructField, StringType,
                                LongType, DoubleType, BooleanType)

SCHEMA = StructType([
    StructField("run_id",                   StringType(),  True),
    StructField("run_ts",                   StringType(),  True),
    StructField("catalog",                  StringType(),  True),
    StructField("schema",                   StringType(),  True),
    StructField("table_name",               StringType(),  True),
    StructField("column_name",              StringType(),  True),
    StructField("layer",                    StringType(),  True),
    StructField("rule_ref",                 StringType(),  True),
    StructField("classification",           StringType(),  True),
    StructField("affected_count",           LongType(),    True),
    StructField("affected_pct",             DoubleType(),  True),
    StructField("total_rows",               LongType(),    True),
    StructField("sample_values",            StringType(),  True),
    StructField("recommended_action",       StringType(),  True),
    StructField("dictionary_strategy_hint", StringType(),  True),
    StructField("standardization_rule",     StringType(),  True),
    StructField("confidence",               StringType(),  True),
    StructField("needs_steward_review",     BooleanType(), True),
    StructField("decided_action",           StringType(),  True),
    StructField("decided_by",               StringType(),  True),
])

out_fq  = f"`{CFG['out_catalog']}`.`{CFG['out_schema']}`.`{CFG['out_table']}`"
out_tbl = f"{CFG['out_catalog']}.{CFG['out_schema']}.{CFG['out_table']}"

findings_df = spark.createDataFrame(all_findings or [], schema=SCHEMA)

if all_findings:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CFG['out_catalog']}`.`{CFG['out_schema']}`")
    findings_df.write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(out_tbl)
    print(f"ok  {len(all_findings):,} finding(s) written to {out_fq}")
else:
    print("No findings generated.")

print(f"Run ID: {RUN_ID}")


In [0]:
BLOCKING = {
    "MISSING_CONTEXT_ID", "CONTEXT_ID_NULL_ROWS", "SOURCE_KEY_ALL_NULL_ROWS",
    "SOURCE_KEY_HIGH_NULL", "MISSING_DATA_SOURCE_ID", "DATA_SOURCE_ID_NULL_ROWS",
    "DATA_SOURCE_UNREGISTERED", "LAYER_VIOLATION", "PERSON_ID_IN_SOURCE_KEY",
    "ENTERPRISE_ID_IN_SOURCE_KEY",
}

if not all_findings:
    print("No source tracking findings. All tables appear compliant.")
else:
    fdf = findings_df

    # Section 1 -- Blocking
    block_df = fdf.filter(F.col("classification").isin(BLOCKING)) \
                  .orderBy(F.col("affected_pct").desc())
    n_block = block_df.count()
    print("=" * 70)
    print(f"  SECTION 1 -- BLOCKING FINDINGS (certification blockers): {n_block}")
    print("=" * 70)
    if n_block:
        display(block_df.select(
            "table_name","column_name","layer","classification","rule_ref",
            "affected_count","affected_pct","recommended_action","decided_action","decided_by"
        ))
    else:
        print("  None.")

    # Section 2 -- By classification
    print("\n" + "-" * 70)
    print("SECTION 2 -- Findings by classification")
    print("-" * 70)
    display(
        fdf.groupBy("classification","rule_ref")
           .agg(
               F.count("*").alias("findings"),
               F.countDistinct("table_name").alias("tables"),
               F.sum("affected_count").alias("total_affected"),
           ).orderBy(F.col("findings").desc())
    )

    # Section 3 -- context_id (tenant isolation)
    ctx_df = fdf.filter(F.col("rule_ref") == "§context_id")
    n_ctx  = ctx_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 3 -- context_id / tenant isolation findings ({n_ctx})")
    print("  NULL context_id or missing row filter = certification blocker")
    print("-" * 70)
    if n_ctx:
        display(ctx_df.select(
            "table_name","column_name","classification",
            "affected_count","recommended_action","decided_action","decided_by"
        ).orderBy("table_name","classification"))

    # Section 4 -- data_source_id (provenance)
    dsrc_df = fdf.filter(F.col("rule_ref") == "§data_source_id")
    n_dsrc  = dsrc_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 4 -- data_source_id / provenance findings ({n_dsrc})")
    print("  Must be carried through all layers; unregistered values block load")
    print("-" * 70)
    if n_dsrc:
        display(dsrc_df.select(
            "table_name","column_name","classification",
            "affected_count","sample_values","recommended_action","decided_action","decided_by"
        ).orderBy("table_name","classification"))

    # Section 5 -- source record keys
    key_df = fdf.filter(F.col("rule_ref") == "§source-record-keys")
    n_key  = key_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 5 -- Source record key findings ({n_key})")
    print("  At least one of patient_id/member_id/record_id must be non-NULL per row")
    print("-" * 70)
    if n_key:
        display(key_df.select(
            "table_name","column_name","classification",
            "affected_count","affected_pct","recommended_action","decided_action","decided_by"
        ).orderBy("table_name","classification"))

    # Section 6 -- Conflation risks
    conf_df = fdf.filter(F.col("rule_ref").isin("§conflation"))
    n_conf  = conf_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 6 -- Conflation risks ({n_conf})")
    print("  context_id vs data_source_id conflation; EMPI keys in source columns")
    print("-" * 70)
    if n_conf:
        display(conf_df.select(
            "table_name","column_name","classification",
            "recommended_action","decided_action","decided_by"
        ).orderBy("table_name"))

    # Section 7 -- EMPI readiness summary per table
    print("\n" + "-" * 70)
    print("SECTION 7 -- EMPI readiness per table")
    print("  Composite key: (data_source_id, context_id, source_keys) must be unique and complete")
    print("-" * 70)
    empi_summary = []
    for tbl, inv in table_inventory.items():
        has_ctx  = inv.get("ctx_col") is not None
        has_dsrc = inv.get("dsrc_col") is not None
        has_keys = len(inv.get("src_key_present", {})) > 0
        empi_ready = has_ctx and has_dsrc and has_keys and inv.get("src_key_all_null", 0) == 0
        empi_summary.append({
            "table_name":      tbl,
            "has_context_id":  has_ctx,
            "has_data_source_id": has_dsrc,
            "has_source_keys": has_keys,
            "source_keys_present": list(inv.get("src_key_present", {}).keys()),
            "all_null_rows":   inv.get("src_key_all_null", 0),
            "composite_dups":  inv.get("composite_dup", 0),
            "empi_ready":      empi_ready,
            "is_multitenant":  inv.get("is_multitenant", "?"),
            "is_multisource":  inv.get("is_multisource", "?"),
        })
    if empi_summary:
        display(spark.createDataFrame(empi_summary).orderBy("empi_ready", "table_name"))

    # Section 8 -- Work queue
    work_df = fdf.filter(F.col("decided_action").isNotNull())
    n_work  = work_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 8 -- Engineer work queue ({n_work} decided)")
    print("-" * 70)
    if n_work:
        display(work_df.select(
            "table_name","column_name","classification",
            "decided_action","decided_by","standardization_rule"
        ).orderBy("table_name"))
    else:
        print("  No decisions recorded yet.")
        print(f"  Query: SELECT * FROM {CFG['out_catalog']}.{CFG['out_schema']}.{CFG['out_table']}")
        print(f"  WHERE run_id = '{RUN_ID}' AND decided_action IS NULL")

    print("\n" + "=" * 70)
    print(f"  Run: {RUN_ID}")
    print(f"  Scope: {cat}.{sch}  |  Layer: {CFG['layer']}")
    print(f"  Tables evaluated: {len(table_inventory)}")
    print(f"  Findings: {len(all_findings):,}  |  Blocking: {n_block}  |  Tenant: {n_ctx}  |  Provenance: {n_dsrc}  |  Keys: {n_key}")
    print("=" * 70)
    print("\nDetection-only. No source data was modified.")
    print("Every finding requires data steward review before any action.")
